<a href="https://colab.research.google.com/github/luiszn/ETL_ElOferton/blob/master/ElOferton_Experimentacion_Definitiva_Spark_RDD_SQL_MLlib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experimentación integral de Big Data con Apache Spark en El Ofertón
### Google Colab · RDD · Spark SQL · MLlib · RE6

**Caso de estudio:** análisis de ventas, inventario comercial y comportamiento de compra de El Ofertón.

**Fuente principal:** `ElOferton_Operaciones_BI_BigData.csv` (archivo operativo adjunto).

> **Propósito:** desarrollar una experimentación reproducible con los datos reales suministrados, aplicando las técnicas de las guías de Spark inicial, Semana 12 (RDD) y RE6 (Spark SQL y MLlib). El notebook genera resultados dinámicamente; no contiene resultados de ejecución inventados.

---

## Cómo utilizar este notebook

1. Abre el archivo `.ipynb` en Google Colab.
2. Selecciona un entorno de ejecución **Python 3 con CPU**.
3. Ejecuta las celdas en orden, desde el inicio.
4. Cuando aparezca el selector de archivos, carga `ElOferton_Operaciones_BI_BigData.csv`.
5. Conserva la misma sesión y los mismos recursos durante las mediciones SQL.
6. Al finalizar, descarga el ZIP de evidencias generado por el notebook.

**Importante:** Google Colab ejecutará Spark en `local[2]`. Es procesamiento paralelo local, no un clúster real con nodos independientes. Por ello, las conclusiones de rendimiento se limitan al entorno medido.

## 0. Diseño previo del experimento (completar antes de medir)

La guía RE6 solicita definir ocho elementos del protocolo antes de conocer los resultados.

| Elemento | Definición para esta experimentación |
|---|---|
| Objetivo | Analizar el comportamiento comercial de El Ofertón y comparar configuraciones de procesamiento y alternativas predictivas. |
| Escenario | La gerencia necesita conocer ventas completadas, margen estimado, categorías, canales y patrones de cantidad vendida. |
| Dataset | CSV operativo de El Ofertón. Se inspeccionarán estructura, calidad, duplicados y valores faltantes. |
| Variables independientes | Volumen de datos, particiones de shuffle y algoritmo predictivo. |
| Variables dependientes | Segundos de consulta, filas por segundo, RMSE y MAE de la predicción de cantidad. |
| Controles | Mismo archivo, misma consulta, mismos filtros, semilla fija, mismo conjunto de prueba y misma sesión de Colab. |
| Herramientas | Google Colab, Python, PySpark, Spark SQL, RDD, MLlib, Pandas, NumPy, SciPy y Matplotlib. |
| Procedimiento | Inspección, calidad, RDD, análisis SQL, benchmark controlado, análisis estadístico, MLlib y exportación de evidencias. |

### Hipótesis que se evaluarán

- **SQL:** las configuraciones de 2, 8 y 32 particiones pueden producir tiempos diferentes, pero deben generar exactamente los mismos resultados agregados.
- **MLlib:** la regresión lineal y Random Forest se compararán contra una referencia que predice la media de `Cantidad` del entrenamiento.
- **Regla práctica:** se reportará si Random Forest reduce al menos 10 % el RMSE frente a la regresión lineal y supera la referencia promedio. Es un umbral didáctico, no una garantía.

**Antes de ejecutar:** escribe tu resultado esperado y una amenaza a la validez que debas controlar.

- Resultado esperado: __________________________________________
- Amenaza a la validez: _________________________________________
- Cómo la controlarás: __________________________________________

## 1. Configuración del entorno en Google Colab

Se fija PySpark 4.0.1 para mantener una versión reproducible, siguiendo la configuración del laboratorio RE6. La guía inicial utiliza `local[*]`; para la comparación controlada de RE6 se utilizará `local[2]`.

In [1]:
%pip -q install pyspark==4.0.1 scipy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
import os
import sys
import re
import csv
import json
import time
import math
import glob
import shutil
import hashlib
import platform
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib.pyplot as plt

import pyspark
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.storagelevel import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

from IPython.display import display, Markdown
from google.colab import files

SEED = 2026
PARTICIONES = [2, 8, 32]
BLOQUES = 12
N_PERM = 4999
UMBRAL_SPEEDUP = 1.10
UMBRAL_REDUCCION_RMSE = 0.10
SPLIT_SEED = 77

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
ROOT = Path.cwd() / f"evidencias_ElOferton_RE6_{RUN_ID}"
for carpeta in ["datos", "tablas", "figuras", "planes", "modelos", "logs", "rdd"]:
    (ROOT / carpeta).mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("PySpark:", pyspark.__version__)
print("Sistema:", platform.platform())
print("CPU lógicas visibles:", os.cpu_count())
print("RAM visible (si está disponible):")
try:
    import psutil
    print(round(psutil.virtual_memory().total / (1024**3), 2), "GB")
except Exception:
    print("No fue posible consultar la RAM desde esta celda.")

java = subprocess.run(["java", "-version"], capture_output=True, text=True)
java_info = java.stderr or java.stdout
print("Java:")
print(java_info)
match_java = re.search(r'version "(\d+)', java_info)
if not match_java or int(match_java.group(1)) not in (17, 21):
    raise RuntimeError("Este laboratorio RE6 requiere Java 17 o 21. Cambia el entorno de Colab y reinicia la sesión.")

Python: 3.13.18
PySpark: 4.0.1
Sistema: Linux-6.6.159+-x86_64-with-glibc2.39
CPU lógicas visibles: 2
RAM visible (si está disponible):
12.67 GB
Java:
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)



## 2. Crear SparkSession y SparkContext

- **SparkSession:** punto de entrada para DataFrames y Spark SQL.
- **SparkContext:** punto de entrada para trabajar directamente con RDD.
- **Driver:** el notebook coordina la ejecución.
- **Executor/Worker:** en Colab `local[2]`, los procesos de trabajo se ejecutan en el mismo entorno.
- **Particiones:** unidades lógicas de trabajo que Spark puede procesar en paralelo.

In [4]:
spark = (
    SparkSession.builder
    .appName("ElOferton_Experimentacion_RE6")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.ui.showConsoleProgress", "true")
    .getOrCreate()
)
sc = spark.sparkContext
spark.sparkContext.setLogLevel("ERROR")

print("Aplicación:", sc.appName)
print("Master:", sc.master)
print("Versión Spark:", spark.version)
print("Paralelismo por defecto:", sc.defaultParallelism)
print("Particiones shuffle iniciales:", spark.conf.get("spark.sql.shuffle.partitions"))

Aplicación: ElOferton_Experimentacion_RE6
Master: local[2]
Versión Spark: 4.0.1
Paralelismo por defecto: 2
Particiones shuffle iniciales: 8


## 3. Cargar el CSV operativo de El Ofertón

La guía inicial de Spark muestra cómo cargar un archivo desde Colab y leerlo con Spark. En este caso el archivo disponible es CSV separado por comas, no un Excel.

**No cambies el archivo por un dataset de ejemplo.** La experimentación utiliza los registros operativos adjuntos.

In [ ]:
uploaded = files.upload()
csv_candidates = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_candidates:
    raise ValueError("No se cargó ningún archivo CSV. Vuelve a ejecutar esta celda y selecciona el archivo de El Ofertón.")

CSV_PATH = csv_candidates[0]
print("Archivo seleccionado:", CSV_PATH)
print("Tamaño (MB):", round(os.path.getsize(CSV_PATH) / (1024**2), 2))

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .option("sep", ",")
    .option("quote", '"')
    .option("escape", '"')
    .csv(CSV_PATH)
)

print("Columnas:", len(df_raw.columns))
print("Registros:", df_raw.count())
df_raw.show(5, truncate=False)

## 4. Comprender el dataset y su estructura

### Diccionario de variables observado en el archivo

| Grupo | Columnas |
|---|---|
| Identificadores | `VentaID`, `DetalleVentaID`, `ProductoID`, `SKU`, `ClienteID`, `EmpleadoID`, `AlmacenID`, `ProveedorReferenciaID` |
| Tiempo | `FechaVenta`, `Anio`, `Mes`, `Dia` |
| Producto | `NombreProducto`, `Categoria`, `Marca` |
| Operación | `TipoVenta`, `CanalVenta`, `EstadoVenta`, `MetodoPago` |
| Métricas | `Cantidad`, `CostoCompraUnitario`, `PrecioVentaUnitario`, `Subtotal`, `Descuento`, `TotalLinea`, `CostoTotal`, `MargenBrutoEstimado` |

El CSV inspeccionado contiene 27 columnas y 457 500 filas originales. El notebook vuelve a calcular estas cantidades al ejecutarse para evitar depender de cifras copiadas manualmente.

In [ ]:
print("ESQUEMA DEL DATASET")
df_raw.printSchema()

print("DIMENSIONES")
print("Filas:", df_raw.count())
print("Columnas:", len(df_raw.columns))
print("Nombres:", df_raw.columns)

print("ESTADÍSTICAS DESCRIPTIVAS")
df_raw.describe().show(truncate=False)

## 5. Auditoría de calidad: nulos, duplicados y consistencia

### Decisión metodológica sobre duplicados

La inspección previa del archivo detectó filas exactamente repetidas y algunos `DetalleVentaID` repetidos con valores distintos.

- Las **filas idénticas en todas sus columnas** se pueden retirar para evitar duplicar exactamente la misma observación.
- Los registros con el mismo `DetalleVentaID` pero con información diferente **no se eliminan automáticamente**. Se contabilizan como conflictos para que puedan revisarse con trazabilidad.
- `VentaID` se repite porque una venta puede tener varias líneas de detalle; no se debe tratar como identificador único de cada fila.

Los registros anulados y pendientes tampoco se confundirán con ventas completadas en los indicadores financieros.

In [ ]:
n_original = df_raw.count()
n_columnas = len(df_raw.columns)

# Conteo de nulos por columna.
nulos_expr = [
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in df_raw.columns
]
nulos = df_raw.agg(*nulos_expr).toPandas().T.reset_index()
nulos.columns = ["columna", "valores_nulos"]
nulos["porcentaje_nulos"] = 100 * nulos["valores_nulos"] / max(n_original, 1)
display(nulos.sort_values("valores_nulos", ascending=False))

# Duplicados exactos: todas las columnas deben coincidir.
df_sin_exactos = df_raw.dropDuplicates()
n_sin_exactos = df_sin_exactos.count()
duplicados_exactos = n_original - n_sin_exactos

# Duplicados de clave después de retirar solo filas idénticas.
claves_repetidas = (
    df_sin_exactos.groupBy("DetalleVentaID")
    .count()
    .filter(F.col("count") > 1)
)
n_claves_conflictivas = claves_repetidas.count()
filas_en_claves_conflictivas = (
    claves_repetidas.agg(F.sum("count").alias("n")).first()["n"] or 0
)

print("Filas originales:", n_original)
print("Duplicados exactos retirables:", duplicados_exactos)
print("Filas después de retirar duplicados exactos:", n_sin_exactos)
print("DetalleVentaID con más de una versión no idéntica:", n_claves_conflictivas)
print("Filas involucradas en claves repetidas no idénticas:", filas_en_claves_conflictivas)

# Validaciones básicas: se reportan, no se eliminan sin justificación.
for c in ["Cantidad", "CostoCompraUnitario", "PrecioVentaUnitario", "TotalLinea", "CostoTotal", "MargenBrutoEstimado"]:
    if c in df_raw.columns:
        print(f"{c}: valores negativos =", df_raw.filter(F.col(c) < 0).count())

print("Estados de venta:")
df_raw.groupBy("EstadoVenta").count().orderBy(F.desc("count")).show()

## 6. Preparación de datos y criterios de negocio

### Criterios aplicados

1. Se eliminan únicamente duplicados exactos de todas las columnas.
2. Se conservan los `DetalleVentaID` repetidos con datos diferentes y se deja evidencia de su cantidad.
3. Se crea una vista de ventas completadas para calcular ingresos y márgenes.
4. Los estados `Anulada` y `Pendiente` se conservan para análisis operativo, pero no se suman como ventas completadas.
5. Los valores faltantes de campos descriptivos se conservan en la auditoría y se agrupan como `SIN_DATO` cuando se presentan resultados.
6. No se imputan importes financieros ni cantidades para fabricar indicadores.

**Advertencia:** `MargenBrutoEstimado` es una métrica que viene calculada en el archivo. Se analizará como margen estimado, no como utilidad contable auditada.

In [ ]:
df_clean = df_sin_exactos

# Fecha de venta: se convierte de forma tolerante para poder analizar el tiempo.
df_clean = df_clean.withColumn(
    "FechaVenta_dt",
    F.to_date(F.col("FechaVenta").cast("string"), "yyyy-MM-dd")
)

# Campos descriptivos para agrupar sin perder registros nulos.
for c in ["Categoria", "Marca", "CanalVenta", "MetodoPago", "TipoVenta", "EstadoVenta"]:
    df_clean = df_clean.withColumn(c + "_grupo", F.coalesce(F.col(c).cast("string"), F.lit("SIN_DATO")))

df_clean.createOrReplaceTempView("operaciones_oferton")
df_completadas = df_clean.filter(F.col("EstadoVenta") == "Completada")

print("Registros limpios (sin duplicados exactos):", df_clean.count())
print("Registros completados:", df_completadas.count())
print("Rango de fechas:")
df_clean.agg(F.min("FechaVenta_dt").alias("fecha_min"), F.max("FechaVenta_dt").alias("fecha_max")).show()

## 7. Exploración de datos (EDA) y análisis comercial con Spark SQL

En esta fase se aplican las operaciones de agrupación y agregación mostradas en la guía inicial: `groupBy`, `agg`, `sum`, `count`, `orderBy` y vistas temporales SQL.

### 7.1 Ventas completadas por año

In [ ]:
ventas_anio = (
    df_completadas.groupBy("Anio")
    .agg(
        F.sum("TotalLinea").alias("VENTA_COMPLETADA"),
        F.sum("CostoTotal").alias("COSTO_TOTAL"),
        F.sum("MargenBrutoEstimado").alias("MARGEN_ESTIMADO"),
        F.countDistinct("VentaID").alias("VENTAS_DISTINTAS"),
        F.count("*").alias("LINEAS")
    )
    .withColumn("MARGEN_PCT", F.when(F.col("VENTA_COMPLETADA") != 0,
                                    100 * F.col("MARGEN_ESTIMADO") / F.col("VENTA_COMPLETADA")))
    .orderBy("Anio")
)
ventas_anio.show(truncate=False)

### 7.2 Ventas y margen estimado por categoría

In [ ]:
ventas_categoria = (
    df_completadas.groupBy("Categoria_grupo")
    .agg(
        F.sum("TotalLinea").alias("VENTA_COMPLETADA"),
        F.sum("MargenBrutoEstimado").alias("MARGEN_ESTIMADO"),
        F.sum("Cantidad").alias("UNIDADES"),
        F.count("*").alias("LINEAS")
    )
    .orderBy(F.desc("VENTA_COMPLETADA"))
)
ventas_categoria.show(20, truncate=False)

### 7.3 Canales de venta, tipo de venta y métodos de pago

In [ ]:
print("Ventas por canal:")
df_completadas.groupBy("CanalVenta_grupo").agg(
    F.sum("TotalLinea").alias("VENTA_COMPLETADA"),
    F.countDistinct("VentaID").alias("VENTAS")
).orderBy(F.desc("VENTA_COMPLETADA")).show(truncate=False)

print("Ventas por tipo:")
df_completadas.groupBy("TipoVenta_grupo").agg(
    F.sum("TotalLinea").alias("VENTA_COMPLETADA"),
    F.sum("Cantidad").alias("UNIDADES")
).orderBy(F.desc("VENTA_COMPLETADA")).show(truncate=False)

print("Ventas por método de pago:")
df_completadas.groupBy("MetodoPago_grupo").agg(
    F.sum("TotalLinea").alias("VENTA_COMPLETADA"),
    F.count("*").alias("LINEAS")
).orderBy(F.desc("VENTA_COMPLETADA")).show(truncate=False)

### 7.4 Evolución mensual y productos con mayor venta

In [ ]:
ventas_mes = (
    df_completadas.groupBy("Anio", "Mes")
    .agg(
        F.sum("TotalLinea").alias("VENTA_COMPLETADA"),
        F.sum("MargenBrutoEstimado").alias("MARGEN_ESTIMADO"),
        F.sum("Cantidad").alias("UNIDADES")
    )
    .orderBy("Anio", "Mes")
)
ventas_mes.show(36, truncate=False)

top_productos = (
    df_completadas.groupBy("SKU", "NombreProducto")
    .agg(
        F.sum("TotalLinea").alias("VENTA_COMPLETADA"),
        F.sum("Cantidad").alias("UNIDADES"),
        F.sum("MargenBrutoEstimado").alias("MARGEN_ESTIMADO")
    )
    .orderBy(F.desc("VENTA_COMPLETADA"))
)
top_productos.show(15, truncate=False)

### 7.5 Estado de las operaciones

Esta tabla permite distinguir operaciones completadas, anuladas y pendientes. No se debe interpretar el total de los estados anulados o pendientes como ingreso reconocido.

In [ ]:
resumen_estado = (
    df_clean.groupBy("EstadoVenta_grupo")
    .agg(
        F.count("*").alias("LINEAS"),
        F.countDistinct("VentaID").alias("VENTAS_DISTINTAS"),
        F.sum("TotalLinea").alias("IMPORTE_REGISTRADO"),
        F.sum("MargenBrutoEstimado").alias("MARGEN_ESTIMADO_REGISTRADO")
    )
    .orderBy(F.desc("LINEAS"))
)
resumen_estado.show(truncate=False)

## 8. Visualización de resultados comerciales

Se convierten a Pandas únicamente tablas agregadas y acotadas, no el conjunto completo de 450 mil registros. Los gráficos permiten observar tendencias y comparar categorías.

In [ ]:
ventas_anio_pd = ventas_anio.toPandas()
ventas_categoria_pd = ventas_categoria.limit(12).toPandas()
ventas_mes_pd = ventas_mes.toPandas()
top_productos_pd = top_productos.limit(10).toPandas()
canal_pd = (
    df_completadas.groupBy("CanalVenta_grupo")
    .agg(F.sum("TotalLinea").alias("VENTA"))
    .orderBy(F.desc("VENTA"))
    .toPandas()
)

plt.figure(figsize=(9, 4))
plt.bar(ventas_anio_pd["Anio"].astype(str), ventas_anio_pd["VENTA_COMPLETADA"])
plt.title("Ventas completadas por año")
plt.xlabel("Año")
plt.ylabel("Importe de ventas")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(ROOT / "figuras/ventas_anio.png", dpi=150)
plt.show()

plt.figure(figsize=(10, 5))
plt.barh(ventas_categoria_pd["Categoria_grupo"][::-1], ventas_categoria_pd["VENTA_COMPLETADA"][::-1])
plt.title("Ventas completadas por categoría")
plt.xlabel("Importe de ventas")
plt.ylabel("Categoría")
plt.tight_layout()
plt.savefig(ROOT / "figuras/ventas_categoria.png", dpi=150)
plt.show()

plt.figure(figsize=(12, 4))
ventas_mes_pd["periodo"] = ventas_mes_pd["Anio"].astype(str) + "-" + ventas_mes_pd["Mes"].astype(str).str.zfill(2)
plt.plot(ventas_mes_pd["periodo"], ventas_mes_pd["VENTA_COMPLETADA"], marker="o", linewidth=1)
plt.title("Evolución mensual de ventas completadas")
plt.xlabel("Periodo")
plt.ylabel("Importe de ventas")
plt.xticks(rotation=90)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(ROOT / "figuras/ventas_mensuales.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(canal_pd["CanalVenta_grupo"], canal_pd["VENTA"])
plt.title("Ventas completadas por canal")
plt.xlabel("Canal")
plt.ylabel("Importe de ventas")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(ROOT / "figuras/ventas_canal.png", dpi=150)
plt.show()

## 9. Diseño de volúmenes para el experimento RE6

La guía RE6 trabaja con tres escalas. En esta adaptación no se fabricarán pedidos sintéticos ni se duplicarán filas para simular un millón de registros. Se usarán subconjuntos deterministas del CSV real:

- **Pequeño:** hasta 1 000 filas.
- **Mediano:** hasta 100 000 filas.
- **Grande:** todas las filas limpias disponibles.

Los subconjuntos pequeño y mediano se seleccionan mediante un hash determinista de `DetalleVentaID`, con una semilla fija. Son subconjuntos del archivo real, no datasets sintéticos. El tamaño grande depende del número de filas después de retirar duplicados exactos.

Se guardará cada volumen en Parquet para que todas las configuraciones SQL lean el mismo conjunto.

In [ ]:
TAMANOS_SOLICITADOS = {"pequeno": 1_000, "mediano": 100_000, "grande": n_sin_exactos}
volumenes = {}
calidad_volumenes = []

for nombre, solicitado in TAMANOS_SOLICITADOS.items():
    if nombre == "grande":
        df_vol = df_clean
    else:
        df_vol = (
            df_clean
            .withColumn("_orden_hash", F.xxhash64(F.col("DetalleVentaID"), F.lit(SEED)))
            .orderBy("_orden_hash", "DetalleVentaID")
            .limit(min(solicitado, n_sin_exactos))
            .drop("_orden_hash")
        )

    ruta = ROOT / "datos" / nombre
    df_vol.write.mode("overwrite").parquet(str(ruta))
    df_ver = spark.read.parquet(str(ruta))
    n_real = df_ver.count()
    volumenes[nombre] = {"n": n_real, "path": str(ruta)}

    calidad_volumenes.append({
        "volumen": nombre,
        "filas_solicitadas": solicitado,
        "filas_reales": n_real,
        "columnas": len(df_ver.columns),
        "bytes_parquet": sum(p.stat().st_size for p in ruta.rglob("*.parquet")),
        "archivos_parquet": len(list(ruta.rglob("*.parquet")))
    })
    print(nombre, "->", n_real, "filas")

calidad_volumenes_pd = pd.DataFrame(calidad_volumenes)
display(calidad_volumenes_pd)
calidad_volumenes_pd.to_csv(ROOT / "tablas/volumenes.csv", index=False)

## 10. Experimentación RDD (Semana 12)

La guía de Semana 12 trabaja directamente con RDD, no con DataFrames, y demuestra transformaciones, acciones, particiones, persistencia y evaluación diferida (*lazy evaluation*).

En esta adaptación, los RDD se construyen desde el CSV operativo. No existe un archivo de comentarios de clientes adjunto; por ello, `flatMap` se demostrará tokenizando `NombreProducto` como texto descriptivo del catálogo. **No se presentará este ejercicio como análisis de sentimiento ni como comentarios reales de clientes.**

In [ ]:
# Leer el archivo de origen como texto para trabajar a nivel RDD.
raw_rdd = sc.textFile(CSV_PATH)
header_line = raw_rdd.first()
header_names = next(csv.reader([header_line]))
idx = {name: i for i, name in enumerate(header_names)}

print("Encabezado:", header_names)
print("Primeras líneas crudas:")
for line in raw_rdd.take(3):
    print(line)
print("Líneas incluyendo encabezado:", raw_rdd.count())
print("Particiones del RDD crudo:", raw_rdd.getNumPartitions())

def parsear_linea(linea):
    campos = next(csv.reader([linea]))
    def get(nombre):
        pos = idx[nombre]
        return campos[pos].strip() if pos < len(campos) else ""
    def num_float(nombre):
        valor = get(nombre)
        return float(valor) if valor not in ("", "NULL", "null") else 0.0
    def num_int(nombre):
        valor = get(nombre)
        return int(float(valor)) if valor not in ("", "NULL", "null") else 0
    return (
        get("VentaID"),
        get("DetalleVentaID"),
        get("Categoria"),
        get("Marca"),
        get("NombreProducto"),
        get("TipoVenta"),
        get("CanalVenta"),
        get("EstadoVenta"),
        get("MetodoPago"),
        get("ClienteID"),
        num_int("Cantidad"),
        num_float("TotalLinea"),
        num_float("CostoTotal"),
        num_float("MargenBrutoEstimado"),
        get("SKU")
    )

rdd_filas = (
    raw_rdd
    .filter(lambda linea: linea != header_line)
    .map(parsear_linea)
    .distinct()  # Retira únicamente tuplas idénticas, no versiones diferentes de una clave.
)
print("RDD de filas creado. Primeros 3 registros parseados:")
for fila in rdd_filas.take(3):
    print(fila)

### 10.1 Transformaciones y acciones: `map`, `filter`, `count`, `take`, `first`

- `map`: transforma cada línea en una estructura de campos.
- `filter`: selecciona registros según una condición.
- `count`, `take` y `first`: son acciones que ejecutan el trabajo.
- Las transformaciones son perezosas: Spark construye el linaje y ejecuta cuando una acción lo requiere.

In [ ]:
# Solo ventas completadas para los indicadores de ingreso.
rdd_completadas = rdd_filas.filter(lambda x: x[7] == "Completada")
print("Filas completadas:", rdd_completadas.count())
print("Primer registro completado:", rdd_completadas.first())
print("Muestra acotada:")
for fila in rdd_completadas.take(5):
    print(fila)

### 10.2 KPI con `reduceByKey`: ingresos y unidades por categoría

`reduceByKey` combina valores que comparten una clave. Para sumar importes o unidades, evita agrupar todos los registros en listas como ocurriría con `groupByKey`.

In [ ]:
ingresos_categoria_rdd = (
    rdd_completadas
    .filter(lambda x: x[2] != "")
    .map(lambda x: (x[2], x[11]))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)
print("Ventas completadas por categoría (RDD):")
for item in ingresos_categoria_rdd.take(15):
    print(item)

unidades_categoria_rdd = (
    rdd_completadas
    .filter(lambda x: x[2] != "")
    .map(lambda x: (x[2], x[10]))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)
print("Unidades vendidas por categoría:")
print(unidades_categoria_rdd.take(15))

### 10.3 KPI con `aggregateByKey`: ticket promedio por categoría

`aggregateByKey` permite mantener un acumulador con dos componentes: suma de importes y cantidad de líneas. Después se calcula el promedio.

In [ ]:
suma_conteo_categoria = (
    rdd_completadas
    .filter(lambda x: x[2] != "")
    .map(lambda x: (x[2], (x[11], 1)))
    .aggregateByKey(
        (0.0, 0),
        lambda acc, valor: (acc[0] + valor[0], acc[1] + valor[1]),
        lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
    )
)
ticket_promedio_categoria = (
    suma_conteo_categoria
    .mapValues(lambda x: x[0] / x[1] if x[1] else 0.0)
    .sortBy(lambda x: x[1], ascending=False)
)
print("Importe promedio por línea de detalle y categoría:")
print(ticket_promedio_categoria.take(15))

### 10.4 `flatMap`, `distinct` y frecuencia de palabras del catálogo

Se separan palabras de `NombreProducto`. Es un ejercicio de procesamiento de texto descriptivo del catálogo, no de opiniones ni reseñas de clientes.

In [ ]:
palabras_catalogo = (
    rdd_filas
    .flatMap(lambda x: re.findall(r"[a-záéíóúñ]+", (x[4] or "").lower()))
    .filter(lambda palabra: len(palabra) > 2)
)
frecuencia_palabras = (
    palabras_catalogo
    .map(lambda palabra: (palabra, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)
print("Palabras frecuentes en nombres de producto:")
print(frecuencia_palabras.take(20))

categorias_distintas = (
    rdd_filas
    .map(lambda x: x[2])
    .filter(lambda x: x != "")
    .distinct()
)
print("Categorías distintas:", categorias_distintas.collect())

### 10.5 Operaciones de conjuntos: `intersection` y `subtract`

Se comparan los clientes identificados en operaciones completadas y pendientes. Se excluyen identificadores vacíos porque el CSV contiene clientes sin identificación.

In [ ]:
clientes_completadas = (
    rdd_filas.filter(lambda x: x[7] == "Completada" and x[9] != "")
    .map(lambda x: x[9]).distinct()
)
clientes_pendientes = (
    rdd_filas.filter(lambda x: x[7] == "Pendiente" and x[9] != "")
    .map(lambda x: x[9]).distinct()
)

clientes_en_ambos = clientes_completadas.intersection(clientes_pendientes)
clientes_solo_completadas = clientes_completadas.subtract(clientes_pendientes)

print("Clientes con operaciones completadas:", clientes_completadas.count())
print("Clientes con operaciones pendientes:", clientes_pendientes.count())
print("Clientes presentes en ambos grupos:", clientes_en_ambos.count())
print("Clientes solo en completadas:", clientes_solo_completadas.count())
print("Muestra de clientes en ambos:", clientes_en_ambos.take(10))

### 10.6 `join`: relacionar líneas y totales por venta

Se construyen dos RDD resumidos por `VentaID`: uno suma el importe de sus líneas y otro cuenta cuántas líneas contiene. Luego se unen mediante `join`.

In [ ]:
importe_por_venta = (
    rdd_completadas
    .map(lambda x: (x[0], x[11]))
    .reduceByKey(lambda a, b: a + b)
)
lineas_por_venta = (
    rdd_completadas
    .map(lambda x: (x[0], 1))
    .reduceByKey(lambda a, b: a + b)
)
ventas_enriquecidas = importe_por_venta.join(lineas_por_venta)

print("VentaID, (importe total, número de líneas):")
print(ventas_enriquecidas.take(10))

### 10.7 Particionamiento, persistencia y `unpersist`

La guía Semana 12 muestra cómo revisar las particiones y reutilizar un RDD persistido. `persist` puede evitar recomputar transformaciones costosas cuando el mismo RDD se consulta varias veces; `unpersist` libera los recursos.

In [ ]:
print("Particiones RDD original:", rdd_filas.getNumPartitions())
rdd_reparticionado = rdd_filas.repartition(4)
print("Particiones tras repartition(4):", rdd_reparticionado.getNumPartitions())
print("Elementos por partición:", rdd_reparticionado.glom().map(len).collect())

rdd_persistido = rdd_completadas.persist(StorageLevel.MEMORY_AND_DISK)
print("Materialización del RDD persistido:", rdd_persistido.count())
print("Ingresos completados por categoría (reutilización):")
print(
    rdd_persistido
    .map(lambda x: (x[2] if x[2] else "SIN_DATO", x[11]))
    .reduceByKey(lambda a, b: a + b)
    .take(15)
)
rdd_persistido.unpersist()
print("RDD liberado con unpersist().")

### 10.8 `union`, `saveAsTextFile` y evidencia RDD

Se combina una muestra de categorías de ventas completadas y pendientes mediante `union`, se eliminan repetidos con `distinct` y se guarda un resultado acotado con `saveAsTextFile`.

In [ ]:
cats_completadas = rdd_filas.filter(lambda x: x[7] == "Completada").map(lambda x: x[2]).filter(bool)
cats_pendientes = rdd_filas.filter(lambda x: x[7] == "Pendiente").map(lambda x: x[2]).filter(bool)
categorias_union = cats_completadas.union(cats_pendientes).distinct()

salida_rdd = str(ROOT / "rdd" / "categorias_operativas")
if os.path.exists(salida_rdd):
    shutil.rmtree(salida_rdd)
categorias_union.map(lambda x: str(x)).saveAsTextFile(salida_rdd)
print("Archivos generados:", os.listdir(salida_rdd))

# Guardar también un resumen pequeño de ingresos por categoría.
salida_kpi = str(ROOT / "rdd" / "ingresos_categoria")
if os.path.exists(salida_kpi):
    shutil.rmtree(salida_kpi)
ingresos_categoria_rdd.saveAsTextFile(salida_kpi)
print("Archivos KPI:", os.listdir(salida_kpi))

## 11. Experimento A: comparación de Spark SQL con 2, 8 y 32 particiones

### Pregunta experimental

¿Cómo cambia el tiempo de ejecución de una misma consulta de resumen comercial cuando se configura Spark SQL con 2, 8 o 32 particiones de shuffle?

### Consulta controlada

La consulta agrupa las ventas completadas por año, mes y categoría. Calcula líneas, venta completada y margen estimado. El resultado está agregado y acotado; no se ejecuta `collect()` sobre todas las filas operativas.

### Controles

- Mismos Parquet y consulta dentro de cada volumen.
- Mismas columnas, filtros y agregaciones.
- AQE desactivado para que Spark no ajuste automáticamente las particiones comparadas.
- Sin persistir en caché el DataFrame de entrada para el benchmark.
- Un calentamiento por configuración excluido del análisis.
- 12 bloques; en cada bloque se aleatoriza el orden de las tres configuraciones.
- Se verifica que las tres configuraciones produzcan los mismos resultados.

**Número esperado de mediciones:** 3 volúmenes × 3 configuraciones × 12 bloques = 108 tiempos válidos, más 9 calentamientos.

In [ ]:
SQL_BENCH = '''
SELECT
    Anio,
    Mes,
    COALESCE(Categoria, 'SIN_DATO') AS Categoria,
    COUNT(*) AS LINEAS_COMPLETADAS,
    SUM(TotalLinea) AS VENTA_COMPLETADA,
    SUM(MargenBrutoEstimado) AS MARGEN_ESTIMADO
FROM pedidos_bench
WHERE EstadoVenta = 'Completada'
GROUP BY Anio, Mes, COALESCE(Categoria, 'SIN_DATO')
'''

(ROOT / "consulta_benchmark.sql").write_text(SQL_BENCH, encoding="utf-8")

def resultados_equivalentes(a, b):
    a = sorted(a, key=lambda r: (r["Anio"], r["Mes"], r["Categoria"]))
    b = sorted(b, key=lambda r: (r["Anio"], r["Mes"], r["Categoria"]))
    if len(a) != len(b):
        return False
    for x, y in zip(a, b):
        if (x["Anio"], x["Mes"], x["Categoria"], x["LINEAS_COMPLETADAS"]) != (
            y["Anio"], y["Mes"], y["Categoria"], y["LINEAS_COMPLETADAS"]
        ):
            return False
        if not np.allclose(
            [x["VENTA_COMPLETADA"], x["MARGEN_ESTIMADO"]],
            [y["VENTA_COMPLETADA"], y["MARGEN_ESTIMADO"]],
            rtol=1e-9, atol=1e-6, equal_nan=True
        ):
            return False
    return True

rng_orden = np.random.default_rng(SEED)
mediciones_sql = []
resumen_bi_volumen = []

for nombre, info in volumenes.items():
    spark.catalog.clearCache()
    df_bench = spark.read.parquet(info["path"])
    df_bench.createOrReplaceTempView("pedidos_bench")
    referencia = None

    # Calentamiento por configuración: no se registra como medición.
    for p in PARTICIONES:
        spark.conf.set("spark.sql.shuffle.partitions", str(p))
        q = spark.sql(SQL_BENCH)
        salida = q.collect()
        if referencia is None:
            referencia = salida
        assert resultados_equivalentes(referencia, salida), "Los resultados SQL no son equivalentes."
        (ROOT / "planes" / f"{nombre}_p{p}.txt").write_text(
            q._jdf.queryExecution().toString(), encoding="utf-8"
        )

    for fila in referencia:
        resumen_bi_volumen.append({"volumen": nombre, **fila.asDict()})

    for bloque in range(1, BLOQUES + 1):
        orden = rng_orden.permutation(PARTICIONES).tolist()
        for posicion, p in enumerate(orden, start=1):
            spark.conf.set("spark.sql.shuffle.partitions", str(p))
            inicio = datetime.now(timezone.utc).isoformat()
            t0 = time.perf_counter()
            salida = spark.sql(SQL_BENCH).collect()
            segundos = time.perf_counter() - t0
            ok = resultados_equivalentes(referencia, salida)
            assert ok, f"Resultado distinto en {nombre}, bloque {bloque}, p={p}"

            registro = {
                "run_id": RUN_ID, "volumen": nombre, "n": info["n"],
                "bloque": bloque, "orden": posicion, "particiones": p,
                "inicio_utc": inicio, "segundos": segundos,
                "filas_por_s": info["n"] / segundos if segundos > 0 else None,
                "equivalente": ok
            }
            mediciones_sql.append(registro)
            with (ROOT / "logs" / "ejecuciones_SQL.jsonl").open("a", encoding="utf-8") as f:
                f.write(json.dumps(registro, ensure_ascii=False) + "\n")

    print(nombre, "completado:", BLOQUES * len(PARTICIONES), "mediciones")

sql = pd.DataFrame(mediciones_sql)
sql.to_csv(ROOT / "tablas" / "ejecuciones_SQL.csv", index=False)
pd.DataFrame(resumen_bi_volumen).to_csv(ROOT / "tablas" / "BI_resumen_por_volumen.csv", index=False)

assert len(sql) == len(volumenes) * len(PARTICIONES) * BLOQUES
assert sql["equivalente"].all()
display(sql.head(10))

### 11.1 Revisar los planes físicos

Abre los planes guardados en la carpeta `planes`. Busca `Exchange` y `hashpartitioning` para identificar el efecto de `spark.sql.shuffle.partitions`. El plan físico es evidencia técnica de que la configuración se aplicó.

In [ ]:
for p in PARTICIONES:
    print("\n" + "=" * 70)
    print(f"PLAN FÍSICO CON {p} PARTICIONES")
    print("=" * 70)
    spark.conf.set("spark.sql.shuffle.partitions", str(p))
    spark.sql(SQL_BENCH).explain("formatted")

print("Planes guardados:", sorted([p.name for p in (ROOT / "planes").glob("*.txt")])[:10])

### 11.2 Estadística descriptiva y gráficos de rendimiento

- **Mediana:** tiempo central de las mediciones.
- **IQR:** Q3 − Q1, dispersión del 50 % central.
- **CV:** desviación estándar / media × 100.
- **Filas por segundo:** filas de entrada / tiempo.
- **Speedup:** tiempo de la configuración de referencia / tiempo de la alternativa. Un valor mayor que 1 favorece a la alternativa.

In [ ]:
resumen_sql = (
    sql.groupby(["volumen", "n", "particiones"])["segundos"]
    .agg(
        repeticiones="count",
        media_s="mean",
        mediana_s="median",
        sd_s="std",
        q1_s=lambda x: x.quantile(0.25),
        q3_s=lambda x: x.quantile(0.75)
    )
    .reset_index()
)
resumen_sql["IQR_s"] = resumen_sql["q3_s"] - resumen_sql["q1_s"]
resumen_sql["CV_pct"] = 100 * resumen_sql["sd_s"] / resumen_sql["media_s"]
resumen_sql["filas_por_s_mediana"] = resumen_sql["n"] / resumen_sql["mediana_s"]
resumen_sql.to_csv(ROOT / "tablas" / "resumen_SQL.csv", index=False)
display(resumen_sql)

fig, axes = plt.subplots(1, len(volumenes), figsize=(5 * len(volumenes), 4))
if len(volumenes) == 1:
    axes = [axes]
for ax, nombre in zip(axes, volumenes):
    sub = sql[sql.volumen == nombre]
    ax.boxplot(
        [sub[sub.particiones == p].segundos for p in PARTICIONES],
        tick_labels=[str(p) for p in PARTICIONES]
    )
    ax.set(title=nombre, xlabel="Particiones shuffle", ylabel="Segundos")
fig.tight_layout()
fig.savefig(ROOT / "figuras" / "SQL_cajas.png", dpi=160)
plt.show()

fig, axes = plt.subplots(len(volumenes), 1, figsize=(9, 4 * len(volumenes)))
if len(volumenes) == 1:
    axes = [axes]
for ax, nombre in zip(axes, volumenes):
    sub = sql[sql.volumen == nombre]
    for p in PARTICIONES:
        s = sub[sub.particiones == p].sort_values("bloque")
        ax.plot(s.bloque, s.segundos, "o-", label=f"p={p}")
    ax.set(title=nombre, xlabel="Bloque", ylabel="Segundos")
    ax.legend()
fig.tight_layout()
fig.savefig(ROOT / "figuras" / "SQL_bloques.png", dpi=160)
plt.show()

### 11.3 Contraste por bloques e intervalo de confianza

Se utiliza una prueba de aleatorización por bloques sobre rangos, siguiendo la lógica del laboratorio RE6. La comparación primaria será el volumen grande, 2 frente a 8 particiones.

- La prueba global evalúa si hay evidencia de diferencias entre las tres configuraciones dentro de los bloques.
- El ajuste de Holm controla el error familiar al evaluar los tres volúmenes.
- El bootstrap remuestrea bloques para estimar un intervalo de confianza del speedup geométrico p2/p8.
- Los bloques, no las filas individuales, son las unidades de repetición del experimento de rendimiento.

In [ ]:
def prueba_permutacion_bloques(matriz, n_perm=N_PERM, seed=SEED):
    rangos = stats.rankdata(matriz, axis=1, method="average")
    k = rangos.shape[1]

    def estadistico(r):
        return float(np.sum((r.mean(axis=0) - (k + 1) / 2) ** 2))

    observado = estadistico(rangos)
    rng = np.random.default_rng(seed)
    mayores = 0
    for _ in range(n_perm):
        perm = np.array([rng.permutation(fila) for fila in rangos])
        mayores += estadistico(perm) >= observado - 1e-12
    return observado, (mayores + 1) / (n_perm + 1)

pruebas = []
for i, nombre in enumerate(volumenes):
    w = sql[sql.volumen == nombre].pivot(
        index="bloque", columns="particiones", values="segundos"
    )
    matriz = w[PARTICIONES].to_numpy()
    assert not np.isnan(matriz).any()
    est, pval = prueba_permutacion_bloques(matriz, seed=SEED + i)
    pruebas.append({"volumen": nombre, "estadistico_rangos": est, "p_sin_ajuste": pval})

pruebas_pd = pd.DataFrame(pruebas)
orden = np.argsort(pruebas_pd["p_sin_ajuste"].to_numpy())
ajuste = np.zeros(len(pruebas_pd))
acumulado = 0.0
for j, i in enumerate(orden):
    acumulado = max(acumulado, (len(pruebas_pd) - j) * pruebas_pd.loc[i, "p_sin_ajuste"])
    ajuste[i] = min(1.0, acumulado)
pruebas_pd["p_Holm"] = ajuste
pruebas_pd.to_csv(ROOT / "tablas" / "prueba_aleatorizacion_SQL.csv", index=False)
display(pruebas_pd)

# Comparación primaria: grande p2/p8.
w_grande = sql[sql.volumen == "grande"].pivot(
    index="bloque", columns="particiones", values="segundos"
)
d = np.log(w_grande[2].to_numpy() / w_grande[8].to_numpy())
boot = stats.bootstrap(
    (d,), np.mean, n_resamples=9999, method="basic",
    confidence_level=0.95, random_state=SEED
)
speedup = float(np.exp(d.mean()))
li, ls = np.exp([boot.confidence_interval.low, boot.confidence_interval.high])
efecto_sql = pd.DataFrame([{
    "comparacion": "grande p2/p8",
    "bloques": len(d),
    "speedup_geometrico": speedup,
    "IC95_inferior": float(li),
    "IC95_superior": float(ls),
    "cumple_regla_didactica": bool(speedup >= UMBRAL_SPEEDUP and li > 1)
}])
efecto_sql.to_csv(ROOT / "tablas" / "efecto_SQL.csv", index=False)
display(efecto_sql)

## 12. Experimento B: predicción de cantidad vendida con MLlib

### Objetivo

Comparar tres alternativas para estimar `Cantidad` en líneas de ventas completadas:

1. **Promedio:** predice la media de `Cantidad` calculada únicamente con el conjunto de entrenamiento.
2. **Regresión lineal (LR):** modelo lineal de referencia con regularización.
3. **Random Forest (RF):** modelo de árboles que puede representar relaciones no lineales.

### Variables predictoras

- Numéricas: `Anio`, `Mes`, `Dia`.
- Categóricas: `TipoVenta`, `Categoria`, `Marca`, `CanalVenta`, `MetodoPago` y `AlmacenID`.

### Variable objetivo

- `Cantidad`.

**Prevención de fuga de información:** no se utilizarán `TotalLinea`, `Subtotal`, `Descuento`, `CostoTotal` ni `MargenBrutoEstimado` como predictores, porque están relacionados matemáticamente con la cantidad, los precios y los importes calculados. Tampoco se usará `DetalleVentaID` ni `VentaID` como predictor.

El split será aproximadamente 75 % entrenamiento y 25 % prueba, usando hash de `DetalleVentaID` para mantener juntas las versiones repetidas de una misma clave. Los modelos se evalúan sobre el mismo test dentro de cada volumen.

In [ ]:
FEATURE_NUM = ["Anio", "Mes", "Dia"]
FEATURE_CAT = ["TipoVenta", "Categoria", "Marca", "CanalVenta", "MetodoPago", "AlmacenID"]
TARGET = "Cantidad"

metricas_ml = []
split_ml = []

for nombre, info in volumenes.items():
    spark.conf.set("spark.sql.shuffle.partitions", "8")
    datos = spark.read.parquet(info["path"]).filter(F.col("EstadoVenta") == "Completada")

    # Convertir AlmacenID a categoría y sustituir faltantes descriptivos.
    datos = datos.withColumn("AlmacenID", F.col("AlmacenID").cast("string"))
    datos = datos.fillna("NO_ESPECIFICADO", subset=FEATURE_CAT)

    bucket = F.pmod(F.xxhash64(F.col("DetalleVentaID"), F.lit(SPLIT_SEED)), F.lit(100))
    train = datos.filter(bucket < 75)
    test = datos.filter(bucket >= 75)

    ntr, nte = train.count(), test.count()
    assert ntr > 0 and nte > 0 and ntr + nte == datos.count()

    split_ml.append({
        "volumen": nombre, "n_completadas": ntr + nte,
        "n_train": ntr, "n_test": nte, "split_seed": SPLIT_SEED
    })

    # El promedio se aprende únicamente del entrenamiento.
    promedio_train = train.agg(F.avg(TARGET).alias("promedio")).first()["promedio"]

    # Preparación categórica: indexación y One-Hot Encoding.
    index_cols = [c + "_idx" for c in FEATURE_CAT]
    encoded_cols = [c + "_ohe" for c in FEATURE_CAT]
    indexers = [
        StringIndexer(inputCol=c, outputCol=out, handleInvalid="keep")
        for c, out in zip(FEATURE_CAT, index_cols)
    ]
    encoder = OneHotEncoder(
        inputCols=index_cols, outputCols=encoded_cols, handleInvalid="keep"
    )
    assembler = VectorAssembler(
        inputCols=FEATURE_NUM + encoded_cols,
        outputCol="features",
        handleInvalid="keep"
    )

    estimadores = {
        "LR": LinearRegression(
            labelCol=TARGET, featuresCol="features",
            regParam=0.01, elasticNetParam=0.0, maxIter=50
        ),
        "RF": RandomForestRegressor(
            labelCol=TARGET, featuresCol="features",
            numTrees=20, maxDepth=5, maxBins=64, seed=SEED
        )
    }

    # La media, LR y RF comparten exactamente el mismo conjunto de prueba.
    for modelo_nombre, estimador in estimadores.items():
        pipeline = Pipeline(stages=indexers + [encoder, assembler, estimador])

        t0 = time.perf_counter()
        modelo = pipeline.fit(train)
        fit_s = time.perf_counter() - t0

        t0 = time.perf_counter()
        pred = modelo.transform(test).select(
            "DetalleVentaID", TARGET, "prediction"
        )
        metricas = pred.select(
            (F.col("prediction") - F.col(TARGET)).alias("error")
        ).agg(
            F.count("*").alias("n"),
            F.avg(F.pow("error", 2)).alias("mse"),
            F.avg(F.abs("error")).alias("mae")
        ).first()
        eval_s = time.perf_counter() - t0

        rmse = float(math.sqrt(metricas["mse"]))
        mae = float(metricas["mae"])
        check_rmse = RegressionEvaluator(
            labelCol=TARGET, predictionCol="prediction", metricName="rmse"
        ).evaluate(pred)

        assert metricas["n"] == nte
        assert math.isclose(rmse, check_rmse, rel_tol=1e-7, abs_tol=1e-7)

        metricas_ml.append({
            "volumen": nombre, "n_total": ntr + nte,
            "modelo": modelo_nombre, "n_train": ntr, "n_test": nte,
            "RMSE_cantidad": rmse, "MAE_cantidad": mae,
            "fit_s": fit_s, "predict_eval_s": eval_s
        })

        modelo.write().overwrite().save(str(ROOT / "modelos" / f"{nombre}_{modelo_nombre}"))
        print(nombre, modelo_nombre, "RMSE:", round(rmse, 4), "MAE:", round(mae, 4))

    # Evaluación de la referencia promedio.
    pred_promedio = test.select(
        F.col(TARGET),
        F.lit(float(promedio_train)).alias("prediction")
    )
    met = pred_promedio.select(
        (F.col("prediction") - F.col(TARGET)).alias("error")
    ).agg(
        F.count("*").alias("n"),
        F.avg(F.pow("error", 2)).alias("mse"),
        F.avg(F.abs("error")).alias("mae")
    ).first()

    metricas_ml.append({
        "volumen": nombre, "n_total": ntr + nte,
        "modelo": "promedio", "n_train": ntr, "n_test": nte,
        "RMSE_cantidad": float(math.sqrt(met["mse"])),
        "MAE_cantidad": float(met["mae"]),
        "fit_s": 0.0, "predict_eval_s": None
    })

    print(nombre, "-> entrenamiento:", ntr, "| prueba:", nte)

ml = pd.DataFrame(metricas_ml)
split_ml_pd = pd.DataFrame(split_ml)
ml.to_csv(ROOT / "tablas" / "metricas_ML.csv", index=False)
split_ml_pd.to_csv(ROOT / "tablas" / "split_ML.csv", index=False)
display(split_ml_pd)
display(ml.sort_values(["volumen", "modelo"]))

### 12.1 Comparación de métricas predictivas

- **RMSE:** raíz del promedio de errores al cuadrado. Penaliza más los errores grandes.
- **MAE:** promedio del valor absoluto de los errores.
- Ambas métricas se expresan en unidades de `Cantidad`; valores menores indican mejor ajuste en el test.

La comparación de modelos se realiza dentro de cada volumen, utilizando el mismo conjunto de prueba.

In [ ]:
comparaciones_ml = []
for nombre in volumenes:
    v = ml[ml.volumen == nombre].set_index("modelo")
    rmse_lr = v.loc["LR", "RMSE_cantidad"]
    rmse_rf = v.loc["RF", "RMSE_cantidad"]
    rmse_base = v.loc["promedio", "RMSE_cantidad"]
    reduccion = 1 - rmse_rf / rmse_lr if rmse_lr else np.nan

    comparaciones_ml.append({
        "volumen": nombre,
        "reduccion_RMSE_RF_vs_LR_pct": 100 * reduccion,
        "RF_supera_promedio": bool(rmse_rf < rmse_base),
        "cumple_regla_didactica": bool(
            reduccion >= UMBRAL_REDUCCION_RMSE and rmse_rf < rmse_base
        )
    })

comparacion_ml_pd = pd.DataFrame(comparaciones_ml)
comparacion_ml_pd.to_csv(ROOT / "tablas" / "comparacion_ML.csv", index=False)
display(comparacion_ml_pd)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metrica in zip(axes, ["RMSE_cantidad", "MAE_cantidad"]):
    tabla = ml.pivot(index="volumen", columns="modelo", values=metrica)
    tabla = tabla.reindex(list(volumenes))[["promedio", "LR", "RF"]]
    tabla.plot.bar(ax=ax, rot=0)
    ax.set(title=metrica, ylabel="Unidades de cantidad")
fig.tight_layout()
fig.savefig(ROOT / "figuras" / "ML_metricas.png", dpi=160)
plt.show()

## 13. Integración de hallazgos para Business Intelligence

Esta sección conecta los resultados de RDD, Spark SQL y MLlib con decisiones de negocio.

### Indicadores sugeridos para la gerencia

- Venta completada por año, mes, categoría y canal.
- Margen bruto estimado por categoría.
- Unidades vendidas por categoría.
- Distribución de estados de venta.
- Clientes que aparecen tanto en ventas completadas como pendientes.
- Rendimiento de consultas según particiones.
- Error de los modelos que estiman cantidad vendida.

**Cuidado con la interpretación:** `Cantidad` se predice usando fecha y atributos de clasificación disponibles en el dataset. El modelo no incorpora historial individual de producto ni stock disponible, por lo que no debe presentarse como pronóstico completo de inventario ni como sistema de reposición automática.

In [ ]:
# Tabla BI consolidada desde Spark SQL.
bi_categoria = (
    df_completadas.groupBy("Categoria_grupo")
    .agg(
        F.sum("TotalLinea").alias("venta_completada"),
        F.sum("CostoTotal").alias("costo_total"),
        F.sum("MargenBrutoEstimado").alias("margen_estimado"),
        F.sum("Cantidad").alias("unidades"),
        F.countDistinct("VentaID").alias("ventas_distintas")
    )
    .withColumn(
        "margen_pct",
        F.when(F.col("venta_completada") != 0,
               100 * F.col("margen_estimado") / F.col("venta_completada"))
    )
    .orderBy(F.desc("venta_completada"))
)
bi_categoria.toPandas().to_csv(ROOT / "tablas" / "BI_categoria.csv", index=False)
display(bi_categoria)

# Resumen por canal.
bi_canal = (
    df_completadas.groupBy("CanalVenta_grupo")
    .agg(
        F.sum("TotalLinea").alias("venta_completada"),
        F.sum("MargenBrutoEstimado").alias("margen_estimado"),
        F.sum("Cantidad").alias("unidades")
    )
    .orderBy(F.desc("venta_completada"))
)
bi_canal.toPandas().to_csv(ROOT / "tablas" / "BI_canal.csv", index=False)
display(bi_canal)

## 14. Análisis de resultados: preguntas para responder

Completa las respuestas con los valores que aparezcan al ejecutar las celdas. No inventes resultados ni afirmes mejoras que las métricas no respalden.

### Calidad de datos
1. ¿Cuántos registros originales tenía el CSV?
2. ¿Cuántas filas idénticas se retiraron?
3. ¿Cuántos `DetalleVentaID` quedaron con más de una versión diferente?
4. ¿Qué columnas presentan más valores nulos y qué efecto podrían tener?

### RDD
5. ¿Qué categoría generó más venta completada según `reduceByKey`?
6. ¿Qué ventaja aporta `aggregateByKey` para calcular promedios?
7. ¿Qué diferencia existe entre una transformación y una acción?
8. ¿Por qué `collect()` debe utilizarse con cuidado en grandes volúmenes?

### Spark SQL
9. ¿Qué configuración tuvo la menor mediana en cada volumen?
10. ¿La diferencia fue estable entre bloques?
11. ¿Los resultados fueron equivalentes para 2, 8 y 32 particiones?
12. ¿El intervalo de confianza del speedup grande p2/p8 respalda una mejora práctica?
13. ¿Por qué los resultados de `local[2]` no demuestran escalabilidad en un clúster distribuido?

### MLlib
14. ¿Qué modelo obtuvo el menor RMSE y MAE en cada volumen?
15. ¿Random Forest superó al promedio?
16. ¿Redujo al menos 10 % el RMSE frente a la regresión lineal?
17. ¿Qué limitaciones tiene predecir cantidad sin incluir historial por SKU ni disponibilidad de inventario?

### Valor para el negocio
18. ¿Qué categoría y canal concentran mayor venta completada?
19. ¿Qué recomendación concreta se puede sustentar con los resultados?
20. ¿Qué información adicional se necesitaría para convertir el análisis en una solución de inventario predictivo?

## 15. Conclusiones y recomendaciones

Redacta al menos **cuatro conclusiones** y **tres recomendaciones** basadas en los resultados reales.

### Conclusiones
1. Calidad de datos: ____________________________________________
2. Procesamiento RDD / Spark SQL: ______________________________
3. Comparación de rendimiento: _________________________________
4. Evaluación de modelos MLlib: _________________________________

### Recomendaciones
1. Comercial: __________________________________________________
2. Operativa / inventario: ______________________________________
3. Datos / tecnología: __________________________________________

### Limitaciones que deben declararse
- El archivo es un extracto operativo y no contiene necesariamente todos los atributos de inventario.
- Los registros de `DetalleVentaID` repetidos con datos diferentes se conservan para no eliminar información sin una regla de negocio.
- El margen es `MargenBrutoEstimado`, no utilidad contable auditada.
- El experimento Spark se ejecuta en Colab `local[2]`, no en un clúster multinodo.
- Los tiempos medidos dependen de los recursos disponibles y de la carga compartida de Colab.
- El modelo de cantidad no es un pronóstico de demanda por SKU ni una política automática de reposición.

## 16. Exportación de evidencias

Se guardarán tablas CSV, figuras PNG, planes de ejecución, resultados RDD, métricas, modelos y metadatos del entorno. El manifiesto incluirá tamaño y SHA-256 de cada archivo para facilitar la trazabilidad.

Antes de cerrar Spark, verifica que las salidas que necesitas se hayan generado correctamente.

In [ ]:
# Exportar tablas descriptivas que se generaron durante la experimentación.
nulos.to_csv(ROOT / "tablas" / "nulos_por_columna.csv", index=False)
calidad_volumenes_pd.to_csv(ROOT / "tablas" / "volumenes.csv", index=False)
ventas_anio_pd.to_csv(ROOT / "tablas" / "ventas_completadas_anio.csv", index=False)
ventas_categoria_pd.to_csv(ROOT / "tablas" / "ventas_completadas_categoria_top.csv", index=False)
ventas_mes_pd.to_csv(ROOT / "tablas" / "ventas_completadas_mes.csv", index=False)
top_productos_pd.to_csv(ROOT / "tablas" / "top_productos.csv", index=False)
resumen_estado.toPandas().to_csv(ROOT / "tablas" / "resumen_estado.csv", index=False)

entorno = {
    "run_id": RUN_ID,
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "archivo_fuente": os.path.basename(CSV_PATH),
    "filas_originales": int(n_original),
    "columnas": int(n_columnas),
    "duplicados_exactos_retirados": int(duplicados_exactos),
    "detalle_ids_con_versiones_distintas": int(n_claves_conflictivas),
    "filas_limpias": int(n_sin_exactos),
    "volumenes": volumenes,
    "particiones_sql": PARTICIONES,
    "bloques_sql": BLOQUES,
    "semilla": SEED,
    "split_ml_seed": SPLIT_SEED,
    "python": sys.version,
    "pyspark": pyspark.__version__,
    "spark": spark.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "master": sc.master,
    "cpu_logicas": os.cpu_count(),
    "sistema": platform.platform()
}
(ROOT / "entorno.json").write_text(json.dumps(entorno, indent=2, ensure_ascii=False), encoding="utf-8")

# Cierre de Spark.
spark.stop()
print("Sesión Spark detenida.")

In [ ]:
# Crear manifiesto de evidencias y archivo ZIP.
manifest = []
for archivo in sorted(ROOT.rglob("*")):
    if archivo.is_file() and archivo.name != "manifest.json":
        h = hashlib.sha256()
        with archivo.open("rb") as f:
            for bloque in iter(lambda: f.read(1024 * 1024), b""):
                h.update(bloque)
        manifest.append({
            "ruta": str(archivo.relative_to(ROOT)),
            "bytes": archivo.stat().st_size,
            "sha256": h.hexdigest()
        })

(ROOT / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

zip_path = shutil.make_archive(
    str(ROOT), "zip",
    root_dir=ROOT.parent,
    base_dir=ROOT.name
)

print("Carpeta de evidencias:", ROOT)
print("Archivo ZIP:", zip_path)
print("Cantidad de archivos en manifiesto:", len(manifest))
files.download(zip_path)

## 17. Checklist de evidencias frente a las guías y rúbrica

| Evidencia | Dónde se genera |
|---|---|
| SparkSession, SparkContext, master y versión | Sección 2 |
| Carga del CSV y esquema | Secciones 3 y 4 |
| Nulos, duplicados y calidad | Sección 5 |
| Análisis comercial y gráficos | Secciones 7 y 8 |
| RDD desde CSV, `map`, `filter`, `reduceByKey` | Secciones 10.1 y 10.2 |
| `aggregateByKey`, `flatMap`, `distinct` | Secciones 10.3 y 10.4 |
| `intersection`, `subtract`, `join`, `union` | Secciones 10.5 a 10.8 |
| Particiones, `persist`, `unpersist`, `saveAsTextFile` | Secciones 10.7 y 10.8 |
| Benchmark SQL 2, 8 y 32 particiones | Sección 11 |
| 12 bloques, planes, equivalencia y estadísticas | Secciones 11.1 a 11.3 |
| Modelos promedio, LR y RF en tres volúmenes | Sección 12 |
| RMSE, MAE y comparación de modelos | Sección 12.1 |
| Conclusiones y recomendaciones | Sección 15 |
| ZIP, manifiesto, tablas y figuras | Sección 16 |

### Preguntas de defensa individual
- ¿Por qué se utiliza Spark si el archivo puede abrirse con Excel?
- ¿Cuál es la diferencia entre RDD y DataFrame?
- ¿Qué diferencia hay entre transformación y acción?
- ¿Qué significa *lazy evaluation*?
- ¿Por qué `reduceByKey` es adecuado para agregaciones?
- ¿Qué cambia al modificar `spark.sql.shuffle.partitions`?
- ¿Por qué se verifica la equivalencia de resultados antes de comparar tiempos?
- ¿Qué es una fuga de información en Machine Learning?
- ¿Por qué el conjunto de prueba no debe utilizarse para aprender transformaciones?
- ¿Qué limitación tiene ejecutar Spark en Colab con `local[2]`?

**Fin de la guía.** Ejecuta todas las celdas en orden y conserva el notebook junto con el ZIP de evidencias.